In [12]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
filepath="D:/bda_pyspark/datasets/sf-fire-calls.csv"

In [13]:
def create_sparkSession():
    spark=SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark

In [14]:
import pandas as pd
df=pd.read_csv("D:/bda_pyspark/datasets/sf-fire-calls.csv")
df.head()

C:\Users\bda\AppData\Local\Temp\ipykernel_22432\2676057383.py:2: DtypeWarning: Columns (0: StationArea, 1: Box, 2: CallTypeGroup) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("D:/bda_pyspark/datasets/sf-fire-calls.csv")


,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,...,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay
0,20110016,T13,2003235,Structure Fire,01/11/2002,01/10/2002,Other,01/11/2002 01:51:44 AM,2000 Block of CALIFORNIA ST,SF,...,NaN,1,TRUCK,2.0,4.0,5.0,Pacific Heights,"(37.7895840679362, -122.428071912459)",020110016-T13,2.950000
1,20110022,M17,2003241,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 03:01:18 AM,0 Block of SILVERVIEW DR,SF,...,NaN,1,MEDIC,1.0,10.0,10.0,Bayview Hunters Point,"(37.7337623673897, -122.396113802632)",020110022-M17,4.700000
2,20110023,M41,2003242,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 02:39:50 AM,MARKET ST/MCALLISTER ST,SF,...,NaN,1,MEDIC,2.0,3.0,6.0,Tenderloin,"(37.7811772186856, -122.411699931232)",020110023-M41,2.433333
3,20110032,E11,2003250,Vehicle Fire,01/11/2002,01/10/2002,Other,01/11/2002 04:16:46 AM,APPLETON AV/MISSION ST,SF,...,NaN,1,ENGINE,1.0,6.0,9.0,Bernal Heights,"(37.7388432849018, -122.423948785199)",020110032-E11,1.500000
4,20110043,B04,2003259,Alarms,01/11/2002,01/10/2002,Other,01/11/2002 06:01:58 AM,1400 Block of SUTTER ST,SF,...,NaN,1,CHIEF,2.0,4.0,2.0,Western Addition,"(37.7872890372638, -122.424236212664)",020110043-B04,3.483333


In [19]:
def create_dataframe(spark,filepath):
    df=spark.read.csv(filepath,header=True, inferSchema=True)
    df1=df.select('callType','CallDate','City','ZipCode','Neighborhood','Delay')
    return df1

In [23]:
def clean_dataset(df):
    df1=df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2=df1.withColumn('Year',year(col('Date')))\
           .withColumn('Month',month(col('Date')))\
           .withColumn('Week',weekofyear(col('Date')))
    return df2

In [28]:
spark=create_sparkSession()
df=create_dataframe(spark,filepath)
df=clean_dataset(df)
df.printSchema()

root
 |-- callType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- ZipCode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [29]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        callType|City|ZipCode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [46]:
def mapSeason(data):
    if 2<data<6:
        return 'Spring'
    elif 5<data<9:
        return 'Summer'
    elif 8<data<12:
        return 'Autumn'
    else:
        return 'Winter'
seasonUDF=udf(mapSeason, StringType())
clean_df=df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        callType|City|ZipCode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [47]:
clean_df[clean_df['Season'=='Summer']].show()

AnalysisException: [DATATYPE_MISMATCH.FILTER_NOT_BOOLEAN] Cannot resolve "callType" due to data type mismatch: Filter expression "callType" of type "STRING" is not a boolean.;
Filter callType#855: string
+- Project [callType#855, City#861, ZipCode#862, Neighborhood#876, Delay#879, Date#915, Year#929, Month#937, Week#946, mapSeason(Month#937)#1464 AS Season#1465]
   +- Project [callType#855, City#861, ZipCode#862, Neighborhood#876, Delay#879, Date#915, Year#929, Month#937, weekofyear(Date#915) AS Week#946]
      +- Project [callType#855, City#861, ZipCode#862, Neighborhood#876, Delay#879, Date#915, Year#929, month(Date#915) AS Month#937]
         +- Project [callType#855, City#861, ZipCode#862, Neighborhood#876, Delay#879, Date#915, year(Date#915) AS Year#929]
            +- Project [callType#855, City#861, ZipCode#862, Neighborhood#876, Delay#879, Date#915]
               +- Project [callType#855, CallDate#856, City#861, ZipCode#862, Neighborhood#876, Delay#879, to_date(CallDate#856, S